# Explore request costs

MorphIQ SDK for Python

Inspect your account’s rates, quote an option, then choose whether to execute.
All inputs start with synthetic values. Running setup or changing a control sends
no requests. Schedule and Quote each send one zero-CU utility request. Execute sends
one pricing request with your chosen CU ceiling. A quote does not lock a price or
guarantee funding, capacity or numerical success.

Set `MORPHIQ_API_KEY` before opening Jupyter, or enter it in the masked setup prompt.
Use a deployment supporting cost utilities. Quote again after changing inputs.


In [ ]:
import getpass
import os

import ipywidgets as widgets
from IPython.display import display

from morphiq_sdk import APIError, MorphIQ, PriceRequest, TransportError

api_key = os.environ.get("MORPHIQ_API_KEY") or getpass.getpass("MorphIQ API key: ")
spot = widgets.FloatText(value=100.0, description="Spot")
strike = widgets.FloatText(value=100.0, description="Strike")
volatility = widgets.FloatText(value=0.2, description="Volatility")
years = widgets.FloatText(value=1.0, description="Years")
rate = widgets.FloatText(value=0.05, description="Rate")
yield_rate = widgets.FloatText(value=0.0, description="Yield")
style = widgets.Dropdown(options=["european", "american"], description="Exercise")
option_type = widgets.Dropdown(options=["call", "put"], description="Option")
maximum = widgets.IntText(value=1, description="Max CU")
schedule_button = widgets.Button(description="Rates · 1 utility")
quote_button = widgets.Button(description="Quote · 1 utility")
execute_button = widgets.Button(description="Execute · 1 request", disabled=True)
output = widgets.Output()
last_quote = None
quoted_request = None
last_result = None


def current_request():
    return PriceRequest(
        exercise_style=style.value,
        option_type=option_type.value,
        spot=spot.value,
        strike=strike.value,
        volatility=volatility.value,
        time_to_expiry_years=years.value,
        risk_free_rate=rate.value,
        dividend_yield=yield_rate.value,
    )


def show_error(error):
    if isinstance(error, APIError):
        charged = error.metadata.fcu_charged if error.metadata else "unknown"
        print(f"API refused: {error.status_code}; {error.code}; charged CU: {charged}")
    else:
        print(str(error))


def show_schedule(_):
    with output:
        output.clear_output()
        try:
            with MorphIQ(api_key=api_key) as client:
                response = client.cost_schedule(operation="price")
            print(response.schedule.model_dump_json(indent=2))
            print("Utility CU charged:", response.metadata.fcu_charged)
        except (APIError, TransportError, ValueError) as error:
            show_error(error)


def quote_request(_):
    global last_quote, quoted_request
    execute_button.disabled = True
    last_quote = quoted_request = None
    with output:
        output.clear_output()
        try:
            request = current_request()
            with MorphIQ(api_key=api_key) as client:
                response = client.quote(request)
            last_quote, quoted_request = response, request.to_wire()
            maximum.value = response.quote.total_fcu
            execute_button.disabled = False
            print(response.quote.model_dump_json(indent=2))
            print("Selected models:", response.resolved_models)
            print("Utility CU charged:", response.metadata.fcu_charged)
            print("Execution is a separate action. Adjust Max CU if desired.")
        except (APIError, TransportError, ValueError) as error:
            show_error(error)


def execute_request(_):
    global last_result
    with output:
        output.clear_output()
        try:
            request = current_request()
            if last_quote is None or request.to_wire() != quoted_request:
                print("Inputs changed. Quote this request before executing.")
                return
            with MorphIQ(api_key=api_key) as client:
                last_result = client.price(request, max_cu=maximum.value)
            print("Price:", last_result.result.price)
            print("CU charged:", last_result.metadata.fcu_charged)
        except (APIError, TransportError, ValueError) as error:
            show_error(error)


def input_changed(_):
    execute_button.disabled = True


for control in [spot, strike, volatility, years, rate, yield_rate, style, option_type]:
    control.observe(input_changed, names="value")
schedule_button.on_click(show_schedule)
quote_button.on_click(quote_request)
execute_button.on_click(execute_request)
display(
    widgets.VBox(
        [
            widgets.HBox([spot, strike, volatility]),
            widgets.HBox([years, rate, yield_rate]),
            widgets.HBox([style, option_type, maximum]),
            widgets.HBox([schedule_button, quote_button, execute_button]),
            output,
        ]
    )
)